<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    13/05/26  
**Docente**  Iván Carrera

# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

c:\Users\mark_\Documents\ir26a\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [4]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [5]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2417.05it/s]


In [6]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches: 100%|██████████| 4944/4944 [5:31:47<00:00,  4.03s/it]  


In [7]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [9]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_embedding = embed_query(query_text)
query_embedding.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [11]:
# código base para FAISS
import faiss
import numpy as np

# Asumiendo `embeddings` en un array NxD
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# Realizamos una búsqueda a partir de una query
k_resultados = 10
D, I = index.search(query_embedding, k=k_resultados)

# Visualización de los textos recuperados
print(f"Búsqueda para: '{query_text}'\n")
print("-" * 50)

for i in range(k_resultados):
    idx = I[0][i]
    distancia = D[0][i]
    # Mapear el índice recuperado con el chunk de texto correspondiente
    texto_recuperado = chunks_df.iloc[idx]["text"]
    
    print(f"Rank: {i+1} | ID Doc: {idx} | Distancia L2: {distancia:.4f}")
    print(f"Texto: {texto_recuperado[:120]}...\n")

Búsqueda para: 'Battery measuring'

--------------------------------------------------
Rank: 1 | ID Doc: 10176 | Distancia L2: 0.2593
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...

Rank: 2 | ID Doc: 1 | Distancia L2: 0.2764
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...

Rank: 3 | ID Doc: 10177 | Distancia L2: 0.3198
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...

Rank: 4 | ID Doc: 37406 | Distancia L2: 0.3217
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...

Rank: 5 | ID Doc: 71872 | Distancia L2: 0.3228
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...

Rank: 6 | ID Doc: 3740

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.


In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# 1. Levantamos la instancia (en memoria para prototipado)
qdrant_client = QdrantClient(":memory:")
collection_name = "wikipedia_chunks"
vector_size = embeddings.shape[1]

# 2. Creamos la colección
qdrant_client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

# 3. Insertamos datos (id, embedding, payload)
points = []
for i, row in chunks_df.iterrows():
    points.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={"text": row["text"], "doc_id": row["doc_id"]}
        )
    )

# Subimos en lotes para optimizar
qdrant_client.upload_points(collection_name=collection_name, points=points)

# 4. Consultamos Top-k
def qdrant_search(query_vec, k=5):
    search_result = qdrant_client.search(
        collection_name=collection_name,
        query_vector=query_vec.flatten().tolist(),
        limit=k
    )
    results = []
    for hit in search_result:
        results.append({
            "id": hit.id,
            "score": hit.score,
            "text": hit.payload["text"],
            "metadata": {"doc_id": hit.payload["doc_id"]}
        })
    return results

# Ejecución de prueba
res_qdrant = qdrant_search(query_embedding, k=5)
for r in res_qdrant:
    print(f"ID: {r['id']} | Score: {r['score']:.4f} | Text: {r['text'][:60]}...")

### Preguntas
- **¿La métrica usada fue cosine o L2? ¿Por qué?**  
Se utilizó Cosine (Similitud Coseno). Como se normalizaron los embeddings previamente en el modelo E5 (normalize_embeddings=True), calcular el producto punto o la similitud coseno es la forma estándar y más eficiente computacionalmente para medir la distancia angular entre los vectores semánticos.
- **¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?**  
Significativamente más fácil. Qdrant maneja el filtrado nativamente a nivel de motor mediante los payloads. En FAISS puro, la metadata debe manejarse de forma externa (manteniendo diccionarios paralelos) o utilizando *pre-filtering / post-filtering* manual, lo que añade complejidad algorítmica al código.
- **¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?**  
Aumenta ligeramente, pero no de forma lineal respecto al tamaño total del corpus. Al utilizar índices basados en grafos (como HNSW por debajo), la búsqueda es de complejidad logarítmica o sublineal, por lo que recuperar un $k$ mayor solo impacta en la fase final de recolección y ordenamiento de la cola de resultados.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).


In [ ]:
from pymilvus import MilvusClient

# 1. Conectamos a Milvus (versión Lite local)
milvus_client = MilvusClient("milvus_demo.db")
collection_name = "wiki_collection"

# 2. Creamos esquema y colección
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embeddings.shape[1],
    metric_type="COSINE" # o IP (Inner Product)
)

# 3. Insertamos datos
data = []
for i, row in chunks_df.iterrows():
    data.append({
        "id": i,
        "vector": embeddings[i].tolist(),
        "text": row["text"],
        "doc_id": row["doc_id"]
    })

# Insertamos los datos en Milvus
milvus_client.insert(collection_name=collection_name, data=data)

# 4. Creamos índice (IVF_FLAT para ANN)
index_params = milvus_client.prepare_index_params()
index_params.add_index(
    field_name="vector", 
    index_type="IVF_FLAT", 
    metric_type="COSINE", 
    params={"nlist": 128}
)
milvus_client.create_index(collection_name=collection_name, index_params=index_params)
milvus_client.load_collection(collection_name=collection_name)

# 5. Función de búsqueda
def milvus_search(query_vec, k=5, nprobe=16):
    res = milvus_client.search(
        collection_name=collection_name,
        data=[query_vec.flatten().tolist()],
        limit=k,
        search_params={"metric_type": "COSINE", "params": {"nprobe": nprobe}},
        output_fields=["text", "doc_id"]
    )
    
    results = []
    for hit in res[0]:
        results.append({
            "id": hit["id"],
            "score": hit["distance"],
            "text": hit["entity"]["text"],
            "metadata": {"doc_id": hit["entity"]["doc_id"]}
        })
    return results

# Experimento rápido
import time
start = time.time()
res_milvus_5 = milvus_search(query_embedding, k=5)
t_5 = time.time() - start

start = time.time()
res_milvus_20 = milvus_search(query_embedding, k=20)
t_20 = time.time() - start

print(f"Tiempo k=5: {t_5:.4f}s | Tiempo k=20: {t_20:.4f}s")

### Preguntas
- **¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**  
Se ajustó nlist en la creación del índice (número de clusters) y nprobe en el momento de la búsqueda (cantidad de clusters a explorar). Un nprobe bajo incrementa la velocidad pero reduce la precisión (exhaustividad).
- **¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**  
Si se compara la salida de Milvus (con nprobe pequeño) frente a la búsqueda exacta de FAISS (Índice FlatL2), se observa que algunos documentos que FAISS devuelve en posiciones inferiores del Top-k pueden no aparecer en Milvus, ya que ANN sacrifica precisión exacta por velocidad, obviando explorar regiones del espacio vectorial consideradas menos probables.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata


In [ ]:
import weaviate
import weaviate.classes as wvc
from weaviate.util import generate_uuid5

# 1. Conectamos (Embedded local)
client = weaviate.connect_to_local() # Requiere Weaviate corriendo en local o docker
# Nota: Si no tienes docker, puedes instanciar un Weaviate Embedded en Linux/Mac
# import weaviate; client = weaviate.WeaviateClient(connection_params=weaviate.connect.ConnectionParams.from_url("http://localhost:8080", grpc_port=50051))

# 2. Definimos esquema (Clase Document)
if client.collections.exists("Document"):
    client.collections.delete("Document")

documents = client.collections.create(
    name="Document",
    properties=[
        wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
    ]
)

# 3. Insertamos objetos
objects_to_insert = []
for i, row in chunks_df.iterrows():
    objects_to_insert.append(
        wvc.data.DataObject(
            properties={"text": row["text"], "doc_id": row["doc_id"]},
            vector=embeddings[i].tolist(),
            uuid=generate_uuid5(str(i))
        )
    )

# Batch insert
documents.data.insert_many(objects_to_insert)

# 4. Consultamos por similitud
def weaviate_search(query_vec, k=5):
    response = documents.query.near_vector(
        near_vector=query_vec.flatten().tolist(),
        limit=k,
        return_metadata=wvc.query.MetadataQuery(distance=True)
    )
    
    results = []
    for obj in response.objects:
        results.append({
            "id": obj.uuid,
            "score": obj.metadata.distance,
            "text": obj.properties["text"],
            "metadata": {"doc_id": obj.properties["doc_id"]}
        })
    return results

res_weaviate = weaviate_search(query_embedding, k=5)
print(res_weaviate[0])

# client.close() # Siempre cerrar conexión al terminar

### Preguntas
- **¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**  
El enfoque de "schema + objetos" es similar a una base de datos orientada a grafos o de documentos (NoSQL). Las entidades son ricas, pueden tener propiedades anidadas y referencias a otras clases, en contraposición al modelo estrictamente tabular y plano relacional.
- **¿Cómo describirías el trade-off de complejidad vs expresividad?**  
Hay un alto nivel de expresividad semántica (permite modelar relaciones complejas y realizar filtros avanzados), pero a costa de una mayor complejidad inicial para definir la ontología, configurar los tipos de datos y manejar el cliente en comparación con soluciones puramente de *key-vector*.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.


2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)


3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.


In [ ]:
import chromadb

# 1. Creamos cliente y colección (persistente local o efímero)
chroma_client = chromadb.Client() # En memoria
collection = chroma_client.create_collection(name="wiki_chroma")

# 2. Preparamos listas
ids = [str(i) for i in chunks_df.index]
vectors = embeddings.tolist()
docs = chunks_df["text"].tolist()
metadatas = [{"doc_id": doc_id} for doc_id in chunks_df["doc_id"]]

# Insertamos datos (Chroma maneja batching internamente)
collection.add(
    ids=ids,
    embeddings=vectors,
    documents=docs,
    metadatas=metadatas
)

# 3. Consultamos Top-k
def chroma_search(query_vec, k=5):
    results = collection.query(
        query_embeddings=query_vec.tolist(),
        n_results=k
    )
    
    out = []
    for i in range(k):
        out.append({
            "id": results['ids'][0][i],
            "score": results['distances'][0][i],
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i]
        })
    return out

res_chroma = chroma_search(query_embedding, k=5)
for r in res_chroma:
    print(f"Chroma Hit -> ID: {r['id']} | Dist: {r['score']:.4f}")

### Preguntas
- **¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**  
Extremadamente fácil. Su API está altamente abstraída y diseñada específicamente para prototipos en Python. Con un solo comando (collection.add) maneja la indexación sin necesidad de configurar parámetros complejos como tamaños de vector, métricas o parámetros ANN de forma explícita.
- **¿Qué limitaciones ves para un sistema en producción?**  
Chroma es estupendo localmente, pero en escenarios empresariales de alta disponibilidad, carece de controles granulares nativos sobre la concurrencia distribuida, gestión de roles/usuarios (RBAC) a nivel de base de datos, y escalabilidad horizontal de nodos que plataformas más consolidadas como Milvus o Qdrant manejan mucho mejor.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata



In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector
import json

# NOTA: Cambia estas credenciales por las de tu servidor Postgres
# conn = psycopg2.connect(dbname="testdb", user="postgres", password="password", host="localhost")

def setup_pgvector(conn, vector_dim):
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    register_vector(conn)
    
    cur.execute("""
        DROP TABLE IF EXISTS documents;
        CREATE TABLE documents (
            id SERIAL PRIMARY KEY,
            text TEXT,
            metadata JSONB,
            embedding vector(%s)
        );
    """, (vector_dim,))
    
    # Inserción
    for i, row in chunks_df.iterrows():
        cur.execute(
            "INSERT INTO documents (id, text, metadata, embedding) VALUES (%s, %s, %s, %s)",
            (i, row["text"], json.dumps({"doc_id": row["doc_id"]}), embeddings[i].tolist())
        )
    conn.commit()
    cur.close()

# setup_pgvector(conn, embeddings.shape[1])

def pgvector_search(conn, query_vec, k=5):
    cur = conn.cursor()
    # Usando el operador <=> para similitud coseno en pgvector
    cur.execute("""
        SELECT id, text, metadata, embedding <=> %s::vector AS distance
        FROM documents
        ORDER BY distance
        LIMIT %s;
    """, (query_vec.flatten().tolist(), k))
    
    rows = cur.fetchall()
    results = []
    for r in rows:
        results.append({
            "id": r[0],
            "text": r[1],
            "metadata": r[2],
            "score": r[3]
        })
    cur.close()
    return results

# res_pg = pgvector_search(conn, query_embedding, k=5)

### Preguntas
- **¿Qué tan “explicable” te parece esta aproximación vs las otras?**  
Es la más explicable y transparente para quienes vienen del mundo tradicional de bases de datos. La operación conceptual $\text{argmin}_{d \in D} \text{dist}(\vec{q}, \vec{d})$ se traduce literalmente a una cláusula SQL comprensible: ORDER BY embedding <=> query LIMIT k
- **¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**  
Permite aprovechar todo el motor relacional (ACID). Se pueden realizar consultas híbridas poderosas cruzando directamente vectores con tablas transaccionales de negocio, usando JOINs estándar, aplicando reglas de negocio estrictas, y ejecutando agregaciones complejas sobre los resultados, todo en una sola transacción.
- **¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**  
Pgvector está limitado por el paradigma relacional generalista de Postgres. Para datasets de miles de millones de vectores o tasas de inserción masivas, Postgres sufrirá cuellos de botella en memoria e I/O frente a motores nativos (Milvus/Qdrant) que están diseñados desde cero y particionados específicamente para cálculo y almacenamiento tensorial distribuido.